# 01 — Data understanding
Load source files and inspect their shape, columns, and sample records. Place CSVs in `data/raw/` and update `config/config.yaml` to match their names.

In [2]:
"""
Data Loading Engine Module
--------------------------
Handles loading CSV datasets safely with explicit verification checks.
"""

from pathlib import Path
from typing import Dict
import sys
import pandas as pd

for project_root in (Path.cwd(), *Path.cwd().parents):
    source_dir = project_root / "src"
    if (source_dir / "ecommerce_analytics").is_dir():
        sys.path.insert(0, str(source_dir))
        break
else:
    raise FileNotFoundError("Could not locate src/ecommerce_analytics from the notebook working directory.")

from ecommerce_analytics.config import ConfigManager


class DataLoader:
    """Loads and validates input datasets from storage."""

    def __init__(self, config: ConfigManager = None) -> None:
        self.config = config or ConfigManager()
        self.raw_data_path = self.config.get_path("raw_data")

    def load_single_csv(self, filename: str) -> pd.DataFrame:
        """Loads a single raw CSV dataset into a pandas DataFrame."""
        file_path = self.raw_data_path / filename
        if not file_path.exists():
            raise FileNotFoundError(f"Dataset file missing: {file_path}")

        try:
            df = pd.read_csv(file_path)
            return df
        except Exception as e:
            raise IOError(f"Error reading dataset file {filename}: {str(e)}")

    def load_all_datasets(self) -> Dict[str, pd.DataFrame]:
        """Loads all Olist e-commerce datasets into a key-value dictionary."""
        files = {
            "customers": "olist_customers_dataset.csv",
            "orders": "olist_orders_dataset.csv",
            "order_items": "olist_order_items_dataset.csv",
            "order_payments": "olist_order_payments_dataset.csv",
            "order_reviews": "olist_order_reviews_dataset.csv",
            "products": "olist_products_dataset.csv",
            "sellers": "olist_sellers_dataset.csv",
            "category_translation": "product_category_name_translation.csv",
        }

        loaded_data = {}
        for key, fname in files.items():
            path = self.raw_data_path / fname
            if path.exists():
                loaded_data[key] = pd.read_csv(path)
                print(
                    f"Loaded {key}: {loaded_data[key].shape[0]} rows, {loaded_data[key].shape[1]} columns."
                )
            else:
                print(f"Warning: File {fname} not found in {self.raw_data_path}.")

        return loaded_data


if __name__ == "__main__":
    loader = DataLoader()
    print("DataLoader instantiated successfully.")
    

DataLoader instantiated successfully.


In [4]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Adjust path to import package modules cleanly from parent directory
MODULE_PATH = Path("../src").resolve()
if str(MODULE_PATH) not in sys.path:
    sys.path.append(str(MODULE_PATH))

from ecommerce_analytics.config import ConfigManager
from ecommerce_analytics.data_loader import DataLoader
from ecommerce_analytics.validation import DataValidator

# Initialize loader and validator
loader = DataLoader()
validator = DataValidator()
datasets = loader.load_all_datasets()

Loaded customers: 99441 rows, 5 columns.
Loaded orders: 99441 rows, 8 columns.
Loaded order_items: 112650 rows, 7 columns.
Loaded order_payments: 103886 rows, 5 columns.
Loaded order_reviews: 99224 rows, 7 columns.
Loaded products: 32951 rows, 9 columns.
Loaded sellers: 3095 rows, 4 columns.
Loaded category_translation: 71 rows, 2 columns.
Loaded customers: 99441 rows, 5 columns.
Loaded orders: 99441 rows, 8 columns.
Loaded order_items: 112650 rows, 7 columns.
Loaded order_payments: 103886 rows, 5 columns.
Loaded order_reviews: 99224 rows, 7 columns.
Loaded products: 32951 rows, 9 columns.
Loaded sellers: 3095 rows, 4 columns.
Loaded category_translation: 71 rows, 2 columns.


In [5]:
# Check dataset dimensions and missing value metrics
shapes_df = validator.inspect_dataset_shapes()
shapes_df

,dataset,rows,columns,total_missing_cells,duplicate_rows,memory_mb
0,customers,99441,5,0,0,29.62
1,orders,99441,8,4908,0,58.97
2,order_items,112650,7,0,0,39.43
3,order_payments,103886,5,0,0,17.81
4,order_reviews,99224,7,145903,0,42.75
5,products,32951,9,2448,0,6.79
6,sellers,3095,4,0,0,0.66
7,category_translation,71,2,0,0,0.01


In [6]:
# Verify primary keys
pk_df = validator.verify_primary_keys()
display(pk_df)

# Verify referential integrity (foreign keys)
fk_df = validator.check_referential_integrity()
display(fk_df)

,dataset,primary_key,total_rows,unique_key_count,is_unique,has_null_in_pk
0,customers,customer_id,99441,99441,True,False
1,orders,order_id,99441,99441,True,False
2,order_items,"order_id, order_item_id",112650,112650,True,False
3,order_payments,"order_id, payment_sequential",103886,103886,True,False
4,order_reviews,review_id,99224,98410,False,False
5,products,product_id,32951,32951,True,False
6,sellers,seller_id,3095,3095,True,False
7,category_translation,product_category_name,71,71,True,False


,child_table,foreign_key,parent_table,parent_key,orphaned_keys_count,match_percentage
0,orders,customer_id,customers,customer_id,0,100.0
1,order_items,order_id,orders,order_id,0,100.0
2,order_items,product_id,products,product_id,0,100.0
3,order_items,seller_id,sellers,seller_id,0,100.0
4,order_payments,order_id,orders,order_id,0,100.0
5,order_reviews,order_id,orders,order_id,0,100.0
